# Professional Practice X1: Clustering Comparison Framework

Build a comprehensive framework to compare clustering algorithms.

**Goal:** Systematically evaluate K-Means, Hierarchical, DBSCAN, and GMM on multiple datasets.

**Metrics:**
- Silhouette Score
- Davies-Bouldin Index
- Calinski-Harabasz Index
- Runtime performance

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("✅ Loaded!")

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import time

# Generate diverse datasets
datasets = {
    'Blobs': make_blobs(n_samples=500, centers=3, random_state=42)[0],
    'Moons': make_moons(n_samples=500, noise=0.1, random_state=42)[0],
    'Circles': make_circles(n_samples=500, noise=0.05, factor=0.5, random_state=42)[0],
}

# Define algorithms
algorithms = {
    'K-Means': lambda: KMeans(n_clusters=3, random_state=42),
    'Hierarchical': lambda: AgglomerativeClustering(n_clusters=3),
    'DBSCAN': lambda: DBSCAN(eps=0.3, min_samples=5),
    'GMM': lambda: GaussianMixture(n_components=3, random_state=42),
}

# Evaluation framework
results = []

for dataset_name, X in datasets.items():
    for algo_name, algo_factory in algorithms.items():
        algo = algo_factory()
        
        # Time the clustering
        start = time.time()
        if hasattr(algo, 'fit_predict'):
            labels = algo.fit_predict(X)
        else:
            labels = algo.fit(X).predict(X)
        runtime = time.time() - start
        
        # Skip if all points in one cluster or too few clusters
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        if n_clusters < 2:
            continue
        
        # Compute metrics
        silhouette = silhouette_score(X, labels)
        davies_bouldin = davies_bouldin_score(X, labels)
        calinski = calinski_harabasz_score(X, labels)
        
        results.append({
            'Dataset': dataset_name,
            'Algorithm': algo_name,
            'Silhouette': silhouette,
            'Davies-Bouldin': davies_bouldin,
            'Calinski-Harabasz': calinski,
            'Runtime (s)': runtime,
        })

# Create comparison table
df_results = pd.DataFrame(results)
print("\nClustering Algorithm Comparison:")
print("="*80)
print(df_results.to_string(index=False))

# Visualize best algorithm per dataset (by Silhouette)
best_per_dataset = df_results.loc[df_results.groupby('Dataset')['Silhouette'].idxmax()]
print("\n\nBest Algorithm per Dataset (by Silhouette Score):")
print("="*80)
print(best_per_dataset[['Dataset', 'Algorithm', 'Silhouette']].to_string(index=False))

# Heatmap of Silhouette scores
pivot = df_results.pivot(index='Algorithm', columns='Dataset', values='Silhouette')
plt.figure(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGn', cbar_kws={'label': 'Silhouette Score'})
plt.title('Clustering Algorithm Performance Comparison', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

print("\n✅ Comprehensive clustering comparison complete!")